# Bispectrum and bicoherence

## Summary

Bicoherence is a squared, normalized version of the bispectrum. Bicoherence is real and normalized between 0 and 1. Bicoherence measure how much phase coupling / "locking" there is between signals. In the following, we consider auto-bispectrum and auto-bicoherence - i.e. the phase coupling between different frequency components in the same signal.

## Definitions

### Bispectrum

$B(f_1, f_2) = F(f_1) . F(f_2) . F^*(f_1+f_2)$  , with $f_1$ and $f_2$ the 2 frequencies considered, $B$ the bispectrum, $F$ the Fourier transform, $^*$ the complex conjugate.

A $F$ transform results in a complex vector. From standard complex calculations, the $B$ magnitude is the product of the $F$ magnitudes, and the $B$ phase is the sum of the $F$ phases (with a - for the $^*$ one).

## Bicoherence

The bicoherence is obtained by averaging over several bispectra, and normalizing. There are several ways to do the normalization. We will prefer the "absolute norm" normalization:

$b(f_1, f_2) = abs(sum_n(B_n(f_1, f_2)) / sum_n(abs(B_n(f_1, f_2)))$  , with $b$ the bicoherence, $B_n(f_1, f_2)$ the bicoherence spectrum on segment number $n$ at frequencies $f_1$ and $f_2$, $sum_n$ the sum over the $n$ segments.

### Interpretation

Considering a signal that is stationary in time:

- if the signal components at frequencies $f_1$, $f_2$, $f_3$, are not phase locked, then the phase of the $B_n(f_1, f_2)$ will be random, and the $sum_n(B_n(f_1, f_2)$ will average to $0$ at the limit; hence, the bicoherence will be $0$
- if the signal components at frequencies $f_1$, $f_2$, $f_3$ are phase locked, then the phase of the $B_n(f_1, f_2)$ will be constant, and the $abs(sum_n(B_n(f_1, f_2))$ will average to $n . abs(B_n(f_1, f_2))$; so does the denominator, so the bicoherence will be $1$.

So a bicoherence of $0$ indicates that the signals at $f_1$, $f_2$, $f_3$ are not phase locked / coupled, and a bicoherence of $1$ indicate that they are fully phase locked / coupled.

Note that this is a necessary, but not a sufficient, condition for frequency components to be coupled by a non-linear mechanism (like "correlation is not causation").

### Possible extensions

TODO: discuss:

- 2 different signals
- different sampling frequencies

## Sources

- https://en.wikipedia.org/wiki/Bicoherence

In [41]:
import numpy as np
import numpy.typing as npt

from scipy import fftpack

In [231]:
# A verbose, possibly slow, un-optimized implementation of real-signal bispectrum and bi-coherence.
# The goal is correctness and ease-of-reading, not speed!
# Still, we use FFTs in reasonable ways, and we do vectorized operations not loops! :)
# But we do not cache anything, including FFT coeffs when repeating the same FFT many times; we compute same coefficients possibly twice (symmetry not taken into account)


def split_signal_into_segments(signal: npt.NDArray, segment_length: int, n_overlap: int, use_next_fftlength: bool =True) ->  npt.NDArray:
    """Split a signal into segments:
    Arguments:
        - signal: the signal to split, a 1d numpy array
        - segment_length: the (minimum if use_next_fftlength is True) segment length
        - overlap: the int number of overlap samples between 2 consecutive segments
        - use_next_fftlength: True if should use the next segment length that allows fast FFT, False to force the current segment length
    Returns:
        - array_of_signals: such that array_of_signals[0, :] is the first segment of length segment_length, etc"""

    assert isinstance(signal, np.ndarray)
    assert signal.ndim == 1
    assert isinstance(segment_length, int)
    assert isinstance(n_overlap, int)
    assert isinstance (use_next_fftlength, bool)
    
    if use_next_fftlength:
        segment_length = fftpack.next_fast_len(segment_length)

    start_segments = np.arange(0, len(signal)-segment_length, segment_length-n_overlap)
    nbr_segments = len(start_segments)
    array_of_signals = np.full([nbr_segments, segment_length], np.nan)

    for crrt_segment in range(nbr_segments):
        array_of_signals[crrt_segment, :] = signal[start_segments[crrt_segment]: start_segments[crrt_segment]+segment_length]
    
    return array_of_signals


array_of_signals = split_signal_into_segments(np.arange(0, 10, 1), segment_length=4, n_overlap=2, use_next_fftlength=True)
res = np.array(
    [[0., 1., 2., 3.],
     [2., 3., 4., 5.],
     [4., 5., 6., 7.]])
np.testing.assert_allclose(res, array_of_signals)
array_of_signals = split_signal_into_segments(np.arange(0, 12, 1), segment_length=7, n_overlap=5, use_next_fftlength=True)
res = np.array(
    [[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
     [ 3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]])
np.testing.assert_allclose(res, array_of_signals)


def find_first_greater_or_equal_index(value:float, array: npt.NDArray):
    """Find the index of the first element in the array that is greater than the specified value.
    Arguments:
        - array (numpy.ndarray): The array to search.
        - value (float): The value to compare against.
    Returns:
        - int: The index of the first element greater than the specified value, the last index."""

    bool_array = array > value
    if bool_array.any():
        argmax = np.argmax(bool_array)
        return argmax
    else:
        return len(array)-1


def get_f_range_index(f_range, frequencies: npt.NDArray):
    """Get the range of index in frequencies that cover f_range. We assume that frequencies is an array of rfft
    frequencies from rfftfreq. We always skip the 0-frequency and the last frequency.
    Arguments:
        - f_range: the range of frequencies to cover, either [f1, f2] with f1<f2 and both are floats, or None
        - frequencies: the array of frequencies from rfftfreq
    Returns:
        - (idx_min, idx_max): the range of indexes so that frequencies[idx_min:idx_max] cover f_range"""

    assert isinstance(frequencies, np.ndarray)
    assert frequencies.ndim == 1
    np.testing.assert_approx_equal(0.0, frequencies[0])
    
    if f_range is None:
        return (1, len(frequencies)-1)
    else:
        assert isinstance(f_range, list)
        assert len(f_range) == 2
        assert isinstance(f_range[0], float)
        assert isinstance(f_range[1], float)
        assert f_range[0] < f_range[1]
        idx_min = find_first_greater_index(f_range[0], frequencies)
        if idx_min > 1:
            idx_min = idx_min - 1
        idx_min = max(1, idx_min)
        idx_max = find_first_greater_index(f_range[1], frequencies)
        if idx_max < len(frequencies)-2:
            idx_max = idx_max + 1
        idx_max = min(idx_max, len(frequencies)-1)
        return (int(idx_min), int(idx_max))


assert get_f_range_index(None, np.array([0., 1., 2., 3., 4., 5.])) == (1, 5)
assert get_f_range_index([-1.0, 7.0], np.array([0., 1., 2., 3., 4., 5.])) == (1, 5)
assert get_f_range_index([2., 3.], np.array([0., 1., 2., 3., 4., 5.])) == (2, 4)

def compute_auto_bispectrum(signal: npt.NDArray, sample_frequency: float, window=np.hanning, f1_range=None, f2_range=None):
    """Compute a single bispectrum for a real signal.
    Arguments:
        - signal: the signal on which to compute the bispectrum
        - sample_frequency: the sampling frequency
        - window: the windowing algorithm, a window function from
            https://numpy.org/doc/stable/reference/routines.window.html ,
            or None for no window
        - f1_range: the range [min_f1, max_f1] over which take the bisepctrum; if None use all frequencies
        - f2_range: the range [min_f1, max_f2] over which take the bispectrum; if None use all frequencies
    Returns:
        - frequencies: the frequencies of the bispectrum
        - bispectrum: the bispectrum as a np array"""
    
    assert isinstance(signal, np.ndarray)
    assert signal.ndim == 1
    assert np.issubdtype(signal.dtype, np.floating) or np.issubdtype(signal.dtype, np.integer)
    assert isinstance(sample_frequency, float)
    assert sample_frequency > 0.0

    if window is not None:
        window = window(len(signal))
        signal = signal * window

    rfft = np.fft.rfft(signal)
    frequencies = np.fft.rfftfreq(len(signal), 1.0/sample_frequency)

    # print(rfft)
    # print(frequencies)

    (min_index_f1, max_index_f1) = get_f_range_index(f1_range, frequencies)
    (min_index_f2, max_index_f2) = get_f_range_index(f2_range, frequencies)

    indexes_f1 = np.arange(min_index_f1, max_index_f1)
    frequencies_1 = frequencies[indexes_f1]
    rfft1 = rfft[indexes_f1]
    
    indexes_f2 = np.arange(min_index_f2, max_index_f2)
    frequencies_2 = frequencies[indexes_f2]
    rfft2 = rfft[indexes_f2]
    #
    indexes_f2 = np.transpose(indexes_f2[np.newaxis])
    rfft2 = np.transpose(rfft2[np.newaxis])

    bispectrum = rfft1 * rfft2
    
    indexes_f3 = indexes_f1 + indexes_f2
    
    original_shape = indexes_f3.shape
    indexes_f3 = indexes_f3.flatten()
    mask_indexes_f3 = indexes_f3 >= len(rfft)
    indexes_f3[mask_indexes_f3] = 0
    Ff3 = rfft[indexes_f3]
    Ff3[mask_indexes_f3] = np.nan
    Ff3 = np.conjugate(Ff3)
    Ff3 = Ff3.reshape(original_shape)

    # print(bispectrum)
    # print(Ff3)

    bispectrum_out = np.transpose(np.multiply(bispectrum, Ff3))
    
    return frequencies_1, frequencies_2, bispectrum_out


f1, f2, bsp = compute_auto_bispectrum(signal=np.array([1, 2, 3, 2, 1, 1, 2, 3]), sample_frequency=1.0, window=np.hanning, f1_range=None, f2_range=None)
f1_res = np.array([0.125, 0.25 , 0.375])
f2_res = np.array([0.125, 0.25 , 0.375])
bsp_res = np.array([
    [12.96642115-15.52622105j,  3.38154335 -0.93617016j, -0.17457407 -0.58808224j],
    [ 3.38154335 -0.93617016j,  0.2048933  -0.62593076j, np.nan        ],
    [-0.17457407 -0.58808224j,         np.nan        , np.nan        ]
    ])
np.testing.assert_allclose(f1, f1_res)
np.testing.assert_allclose(f2, f2_res)
np.testing.assert_allclose(bsp, bsp_res)

f1, f2, bsp = compute_auto_bispectrum(signal=np.array([1, 2, 3, 2, 1, 1, 2, 3]), sample_frequency=1.0, window=np.hanning, f1_range=[0.1,0.3], f2_range=None)
f1_res = np.array([0.125, 0.25])
f2_res = np.array([0.125, 0.25 , 0.375])
bsp_res = np.array([
    [12.96642115-15.52622105j,  3.38154335 -0.93617016j, -0.17457407 -0.58808224j],
    [ 3.38154335 -0.93617016j,  0.2048933  -0.62593076j, np.nan        ],
    ])
np.testing.assert_allclose(f1, f1_res)
np.testing.assert_allclose(f2, f2_res)
np.testing.assert_allclose(bsp, bsp_res)

f1, f2, bsp = compute_auto_bispectrum(signal=np.array([1, 2, 3, 2, 1, 1, 2, 3]), sample_frequency=1.0, window=np.hanning, f1_range=None, f2_range=[0.1,0.3])
f1_res = np.array([0.125, 0.25 , 0.375])
f2_res = np.array([0.125, 0.25])
bsp_res = np.array([
    [12.96642115-15.52622105j,  3.38154335 -0.93617016j],
    [ 3.38154335 -0.93617016j,  0.2048933  -0.62593076j],
    [-0.17457407 -0.58808224j,         np.nan          ]
    ])
np.testing.assert_allclose(f1, f1_res)
np.testing.assert_allclose(f2, f2_res)
np.testing.assert_allclose(bsp, bsp_res)


def compute_auto_bicoherence(signal: npt.NDArray, sample_frequency: float, segment_length: int, n_overlap: int, use_next_fftlength: bool=True, window=np.hanning, f1_range=None, f2_range=None, method="absolute_norm"):
    assert isinstance(signal, np.ndarray)
    assert signal.ndim == 1
    assert np.issubdtype(signal.dtype, np.floating) or np.issubdtype(signal.dtype, np.integer)
    
    assert isinstance(sample_frequency, float)
    assert sample_frequency > 0.0

    assert isinstance(segment_length, int)
    
    assert isinstance(n_overlap, int)
    
    assert isinstance(use_next_fftlength, bool)

    assert method == "absolute_norm"
    
    # split the signal in segments
    array_of_signals = split_signal_into_segments(signal, segment_length, n_overlap, use_next_fftlength)
    n_segments = array_of_signals.shape[0]
    
    # compute the auto bispectrum on each segment
    # put result in a new np.array
    list_bispectrums = []
    for crrt_segment_index in range(n_segments):
        crrt_segment = array_of_signals[crrt_segment_index, :]
        frequencies_1, frequencies_2, bispectrum_out = compute_auto_bispectrum(crrt_segment, sample_frequency, window, f1_range, f2_range)
        list_bispectrums.append(bispectrum_out)

    array_bispectrums = np.array(list_bispectrums)
    
    # average into the bicoherence with the normalization chose
    if method == "absolute_norm":
        num = np.abs(np.sum(array_bispectrums, axis=0))
        denum = np.sum(np.abs(array_bispectrums), axis=0)
        auto_bicoherence = num / denum
    else:
        raise RuntimeError("Unknown method!")
    
    return frequencies_1, frequencies_2, auto_bicoherence

In [233]:
compute_auto_bicoherence(np.array([1., 2., 4., 3., 1., 6., 7., 1., 2., 4., 5., 6., 7., 3., 4., 1., 2., 5.]), 1.0, 8, 4)

(array([0.125, 0.25 , 0.375]),
 array([0.125, 0.25 , 0.375]),
 array([[0.61250278, 0.89529796, 0.75837889],
        [0.89529796, 0.65817521,        nan],
        [0.75837889,        nan,        nan]]))

In [223]:
a = np.array(
    [[ 0.,  1.],
     [ 2.,  3.]])

b = np.array(
    [[ 0.,  10.],
     [ 100.,  1000.]])

l = [a, b]
la = np.array(l)

In [227]:
np.mean(la, axis=0)

array([[  0. ,   5.5],
       [ 51. , 501.5]])

In [224]:
la

array([[[   0.,    1.],
        [   2.,    3.]],

       [[   0.,   10.],
        [ 100., 1000.]]])

In [218]:
compute_auto_bispectrum(signal=np.array([1, 2, 3, 2, 1, 1, 2, 3]), sample_frequency=1.0, window=np.hanning, f1_range=None, f2_range=[0.1, 0.26])

(array([0.125, 0.25 , 0.375]),
 array([0.125, 0.25 ]),
 array([[12.96642115-15.52622105j,  3.38154335 -0.93617016j],
        [ 3.38154335 -0.93617016j,  0.2048933  -0.62593076j],
        [-0.17457407 -0.58808224j,         nan        +nanj]]))

In [207]:
f1, f2, bsp = compute_auto_bispectrum(signal=np.array([1, 2, 3, 2, 1, 1, 2, 3]), sample_frequency=1.0, window=np.hanning, f1_range=[0.1,0.3], f2_range=None)


In [208]:
bsp.shape

(3, 2)

In [190]:
a0 = 6.04951557+0.j
a1 = -2.46066592-2.63546567j
a2 = -1.25980717+0.9131982j
a3 = 0.55969705+0.27907673j
a4 = 0.2720365 +0.j

In [191]:
a3

(0.55969705+0.27907673j)

In [192]:
a1*a1*np.conjugate(a2)

np.complex128(12.966421083915519-15.526221134549477j)

In [195]:
a1*a3*np.conjugate(a4)

np.complex128(-0.1745740689041467-0.5880822378297776j)

In [184]:
a1*a2

(5.5066670749964395+1.0731028584095101j)

In [158]:
a1*a3

(-0.6417303152486769-2.161776959451315j)

In [164]:
a3*a3

(0.23537696655120965+0.312396845009293j)

In [96]:
a * b

array([[  1,   2,   3],
       [ 10,  20,  30],
       [100, 200, 300]])

In [97]:
np.arange(1, 5)

array([1, 2, 3, 4])

In [98]:
b*a

array([[  1,   2,   3],
       [ 10,  20,  30],
       [100, 200, 300]])

In [105]:
a[[0, 0, 0, 1, 1, 1, np.nan]]

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices